# **Amazon Prime Video – Data Cleaning for Tableau Dashboard**

This notebook cleans the Amazon Prime Video dataset that I used in my EDA project, so that it is ready to be loaded into Tableau.

**Input files:** `titles.csv`, `credits.csv`

**Output files:** `titles_clean.csv`, `title_genres.csv`, `title_countries.csv`, `credits_clean.csv`

## 1. Import Libraries

In [1]:
# Import libraries required for cleaning
import pandas as pd
import numpy as np
import ast

## 2. Load the Datasets

In [3]:
# Load the titles and credits datasets
titles = pd.read_csv('/content/titles.csv')
credits = pd.read_csv('/content/credits.csv')

In [4]:
# Check the size of both datasets before cleaning
print("Titles shape:", titles.shape)
print("Credits shape:", credits.shape)

Titles shape: (9871, 15)
Credits shape: (124235, 5)


## 3. Remove Duplicate Rows

In [5]:
# Count duplicate rows before removing them
print("Duplicate rows in titles:", titles.duplicated().sum())
print("Duplicate rows in credits:", credits.duplicated().sum())

Duplicate rows in titles: 3
Duplicate rows in credits: 56


In [6]:
# Remove duplicate rows from both datasets
titles = titles.drop_duplicates().reset_index(drop=True)
credits = credits.drop_duplicates().reset_index(drop=True)

print("Titles shape after removing duplicates:", titles.shape)
print("Credits shape after removing duplicates:", credits.shape)

Titles shape after removing duplicates: (9868, 15)
Credits shape after removing duplicates: (124179, 5)


In [7]:
# Check that every title id is unique now
print("Duplicate title IDs:", titles['id'].duplicated().sum())

Duplicate title IDs: 0


## 4. Clean the Text Columns

- `type` has the values MOVIE and SHOW, which I changed to Movie and TV Show so they look better in the dashboard.
- `age_certification` has many missing values, so I replaced them with "Not Rated" instead of deleting those rows.
- The score, votes and popularity columns are left empty when missing, because filling them with 0 would give wrong averages.
- `seasons` is empty for movies, which is expected.

In [8]:
# Make the content type labels easier to read
titles['type'] = titles['type'].replace({'MOVIE': 'Movie', 'SHOW': 'TV Show'})

# Replace missing age certification with "Not Rated"
titles['age_certification'] = titles['age_certification'].fillna('Not Rated')

# Replace missing descriptions with blank text
titles['description'] = titles['description'].fillna('')

titles['type'].value_counts()

,count
type,
Movie,8511
TV Show,1357


## 5. Convert Genres and Countries into Lists

In the CSV file, `genres` and `production_countries` are saved as text like `['drama', 'comedy']`. I used `ast.literal_eval()` to convert them into real Python lists, the same way as in my EDA notebook.

In [9]:
# Convert genres and production countries from text to lists
titles['genres'] = titles['genres'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

titles['production_countries'] = titles['production_countries'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

In [10]:
# Some titles have an empty genre or country list
print("Titles with no genre:", (titles['genres'].str.len() == 0).sum())
print("Titles with no country:", (titles['production_countries'].str.len() == 0).sum())

Titles with no genre: 209
Titles with no country: 821


In [11]:
# Label the empty lists as Unknown so these titles are not lost
titles['genres'] = titles['genres'].apply(lambda x: x if len(x) > 0 else ['unknown'])
titles['production_countries'] = titles['production_countries'].apply(lambda x: x if len(x) > 0 else ['Unknown'])

## 6. Create Separate Genre and Country Tables

A title can have more than one genre or country. If I put everything in one table, the title count in Tableau would be wrong. So I made one row per title and genre, and one row per title and country. These tables can be joined back to the main table using `id`.

In [12]:
# Create the genre table (one row for each title and genre)
title_genres = titles[['id', 'genres']].explode('genres')
title_genres = title_genres.rename(columns={'genres': 'genre'})

# Make the genre names look better
title_genres['genre'] = title_genres['genre'].replace({
    'scifi': 'Sci-Fi',
    'documentation': 'Documentary'
})
title_genres['genre'] = title_genres['genre'].str.capitalize()
title_genres['genre'] = title_genres['genre'].replace({'Sci-fi': 'Sci-Fi'})

title_genres.head(10)

,id,genre
0,ts20945,Comedy
0,ts20945,Family
0,ts20945,Animation
0,ts20945,Action
0,ts20945,Fantasy
0,ts20945,Horror
1,tm19248,Action
1,tm19248,Drama
1,tm19248,War
1,tm19248,Western


In [13]:
# Create the country table (one row for each title and country)
title_countries = titles[['id', 'production_countries']].explode('production_countries')
title_countries = title_countries.rename(columns={'production_countries': 'country_code'})

title_countries.head(10)

,id,country_code
0,ts20945,US
1,tm19248,US
2,tm82253,US
3,tm83884,US
4,tm56584,US
5,tm160494,US
6,tm87233,US
7,tm19424,US
8,tm116781,US
9,tm112005,US


In [14]:
# Keep genres and countries as simple text in the main table (useful for tooltips)
titles['genres_list'] = titles['genres'].apply(lambda x: ', '.join(x))
titles['countries_list'] = titles['production_countries'].apply(lambda x: ', '.join(x))

# Number of genres for each title
titles['genre_count'] = titles['genres'].apply(len)

# The list columns are not needed in the main table anymore
titles = titles.drop(columns=['genres', 'production_countries'])

## 7. Create New Columns for the Dashboard

- `release_decade` – to see how the catalog changed decade by decade
- `runtime_band` – Short, Medium or Long
- `rating_gap` – IMDb score minus TMDB score, to compare the two rating systems
- `imdb_band` – Low, Average or High, based on the IMDb score

In [15]:
# Create the decade column from the release year
titles['release_decade'] = (titles['release_year'] // 10) * 10

# Create runtime bands
titles['runtime_band'] = pd.cut(
    titles['runtime'],
    bins=[0, 60, 120, 1000],
    labels=['Short (up to 60 min)', 'Medium (61-120 min)', 'Long (over 120 min)']
)

# Difference between IMDb and TMDB scores
titles['rating_gap'] = (titles['imdb_score'] - titles['tmdb_score']).round(2)

# Create IMDb rating bands
titles['imdb_band'] = pd.cut(
    titles['imdb_score'],
    bins=[0, 5, 7, 10],
    labels=['Low (below 5)', 'Average (5 to 7)', 'High (above 7)'],
    right=False
)

titles[['title', 'release_year', 'release_decade', 'runtime', 'runtime_band', 'imdb_score', 'imdb_band', 'rating_gap']].head()

,title,release_year,release_decade,runtime,runtime_band,imdb_score,imdb_band,rating_gap
0,The Three Stooges,1934,1930,19,Short (up to 60 min),8.6,High (above 7),1.0
1,The General,1926,1920,78,Medium (61-120 min),8.2,High (above 7),0.2
2,The Best Years of Our Lives,1946,1940,171,Long (over 120 min),8.1,High (above 7),0.3
3,His Girl Friday,1940,1940,92,Medium (61-120 min),7.8,High (above 7),0.4
4,In a Lonely Place,1950,1950,94,Medium (61-120 min),7.9,High (above 7),0.3


## 8. Clean the Credits Dataset

In [16]:
# Make the role labels easier to read (ACTOR -> Actor)
credits['role'] = credits['role'].str.title()

# Directors do not have a character name, so fill the blanks
credits['character'] = credits['character'].fillna('')

# Keep only credits that belong to titles in the cleaned titles table
credits = credits[credits['id'].isin(titles['id'])].reset_index(drop=True)

## 9. Final Check Before Saving

In [17]:
# Check the size of all cleaned datasets
print("titles_clean:", titles.shape)
print("title_genres:", title_genres.shape)
print("title_countries:", title_countries.shape)
print("credits_clean:", credits.shape)

titles_clean: (9868, 20)
title_genres: (22269, 2)
title_countries: (11069, 2)
credits_clean: (124179, 5)


In [18]:
# Check the remaining missing values in the titles table
titles.isnull().sum()[titles.isnull().sum() > 0]

,0
seasons,8511
imdb_id,667
imdb_score,1021
imdb_votes,1031
tmdb_popularity,547
tmdb_score,2080
rating_gap,2549
imdb_band,1021


The remaining missing values are kept on purpose:

- `seasons` is empty because movies do not have seasons.
- Scores, votes and popularity are empty where the data was not available. Tableau ignores empty values when it calculates averages.

## 10. Save the Cleaned Files

In [19]:
# Save the cleaned datasets to use in Tableau
titles.to_csv('titles_clean.csv', index=False, encoding='utf-8-sig')
title_genres.to_csv('title_genres.csv', index=False, encoding='utf-8-sig')
title_countries.to_csv('title_countries.csv', index=False, encoding='utf-8-sig')
credits.to_csv('credits_clean.csv', index=False, encoding='utf-8-sig')

print("All 4 cleaned files saved.")

All 4 cleaned files saved.


## Summary

- Removed 3 duplicate rows from titles and 56 from credits.
- Made the type, genre and role labels easier to read.
- Filled missing age certification with "Not Rated" and empty genres and countries with "Unknown".
- Left missing scores, votes, popularity and seasons empty on purpose.
- Split genres and countries into separate tables so counts are correct in Tableau.
- Added the `release_decade`, `runtime_band`, `rating_gap` and `imdb_band` columns.
- Saved 4 cleaned files for the Tableau dashboard.